In [62]:
# imports   
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime as dt
import ta
import datetime

In [63]:
# load data from csv 
original_data = pd.read_csv('../data/processed/BTCUSDT_1m_2024-12-01_to_2025-01-01_cleaned_robust.csv')
volume_data = pd.read_csv('../data/processed/BTCUSDT_1m_2024-12-01_to_2025-01-volume.csv')
dolar_data = pd.read_csv('../data/processed/BTCUSDT_1m_2024-12-01_to_2025-01-01_dollar_bars_dyn.csv')

In [64]:
def normalize_structure(df):
    """
    Asegura que el dataset tenga las columnas necesarias
    y en el formato correcto antes del feature engineering.
    """
    required_cols = ['open_time', 'close_time', 'open', 'high', 'low', 'close', 'volume', 'return']
    for col in required_cols:
        if col not in df.columns:
            df[col] = np.nan  # agrega columna vacía si falta

    # reordena columnas
    df = df[required_cols]
    return df

Para el modelo predictivo hay que tener las mismas features, así que hay que revisar los notebooks, donde se trataron los datos, también hay que tener en cuenta las features que usamos en un anterior proyecto apra darle más info al modelo, están en obsidian explicadas

Hay que quitar lo de close_time, open_time, tienen directamente usarlas como features, al calcular los retornos hay que usar shift(1) correctaemtne para solo usar los datos anteriores al target.

Los features bases para cada dataset son los siguientes, hay que quitarlos dentro del tratamiento de los bars correspondientes dentro de los notebooks donde se limpiaron los datos. Las features son las siguientes: 

['open_time', 'open', 'high', 'low', 'close', 'volume', 'close_time', 'return']


In [65]:
# date object to datetime

def init_date_conversion(data):
    print("dtype original:", data['open_time'].dtype)
    print("primeros 10 valores (raw):")
    display(data['open_time'].head(10))

    if data['open_time'].dtype == object:
        # eliminar espacios, comillas, saltos de línea
        data['open_time'] = data['open_time'].astype(str).str.strip().str.replace('"', '', regex=False).str.replace("'", "", regex=False)
        data['close_time'] = data['close_time'].astype(str).str.strip().str.replace('"', '', regex=False).str.replace("'", "", regex=False)

    sample_vals = data['open_time'].dropna().astype(str).head(50)
    n_digit_samples = sample_vals.apply(lambda x: x.isdigit()).sum()
    print(f"de los primeros 50 valores, {n_digit_samples} parecen solo dígitos (epoch)")

def robust_to_datetime(series):
    # si ya es datetime, devolver
    if np.issubdtype(series.dtype, np.datetime64):
        return series
    
    s = series.copy()
    as_str = s.dropna().astype(str)
    is_all_digits = as_str.str.match(r'^\d+$').mean()  # proporción de strings sólo dígitos
    
    if is_all_digits > 0.5:
        # la mayoría son timestamps numéricos; inferir ms vs s por magnitud
        # convertir a float para inspección del tamaño (usa sample para evitar overflow)
        sample_num = as_str.sample(min(100, len(as_str))).astype(float)
        median_sample = sample_num.median()
        print("mediana de muestra numérica:", median_sample)
        if median_sample > 1e12:  # típico de ms epoch ( > ~10^12)
            print("-> inferido como epoch en MILLISEGUNDOS")
            out = pd.to_datetime(s.astype(float), unit='ms', errors='coerce')
        elif median_sample > 1e9:  # típico de s epoch ( > ~10^9)
            print("-> inferido como epoch en SEGUNDOS")
            out = pd.to_datetime(s.astype(float), unit='s', errors='coerce')
        else:
            # números pequeños: tratar como strings normalmente
            out = pd.to_datetime(s, errors='coerce')
        return out
    else:
        out = pd.to_datetime(s, errors='coerce', utc=False)
        # si demasiados NaT, intentar forzar formatos comunes
        nat_frac = out.isna().mean()
        print(f"frac NaT tras parse directo: {nat_frac:.3f}")
        if nat_frac > 0.2:
            # intentar parse con varias plantillas comunes
            fmts = [
                "%Y-%m-%d %H:%M:%S.%f",
                "%Y-%m-%d %H:%M:%S",
                "%Y-%m-%dT%H:%M:%S.%fZ",
                "%Y-%m-%dT%H:%M:%SZ",
            ]
            for fmt in fmts:
                try:
                    test = pd.to_datetime(s, format=fmt, errors='coerce')
                    nat_frac2 = test.isna().mean()
                    print(f"  intento con format {fmt} => NaT frac: {nat_frac2:.3f}")
                    if nat_frac2 < nat_frac:
                        out = test
                        nat_frac = nat_frac2
                except Exception:
                    pass
        return out

# Aplicar la función a open_time y close_time
def convert_datetime_columns(data):
    data['open_time_parsed'] = robust_to_datetime(data['open_time'])
    data['close_time_parsed'] = robust_to_datetime(data['close_time'])

    n_total = len(data)
    n_nat_open = data['open_time_parsed'].isna().sum()
    n_nat_close = data['close_time_parsed'].isna().sum()
    print(f"open_time -> NaT: {n_nat_open}/{n_total} ({n_nat_open/n_total:.3%})")
    print(f"close_time -> NaT: {n_nat_close}/{n_total} ({n_nat_close/n_total:.3%})")

    if n_nat_open > 0:
        print("Ejemplos de open_time problemáticos:")
        display(data.loc[data['open_time_parsed'].isna(), 'open_time'].head(10))


    if (n_nat_open / n_total) < 0.05:
        data['open_time'] = data['open_time_parsed']
    else:
        print("Advertencia: demasiados NaT en open_time — revisa los ejemplos anteriores.")
        # no sobrescribimos para evitar pérdida de info

    if (n_nat_close / n_total) < 0.05:
        data['close_time'] = data['close_time_parsed']

    # borrar columnas auxiliares
    data = data.drop(columns=[c for c in ['open_time_parsed', 'close_time_parsed'] if c in data.columns])


    data = data.dropna(subset=['open_time'])

    print("Conversión final: dtype open_time =>", data['open_time'].dtype)
    display(data.head())

    return pd.DataFrame(data)


In [66]:
def add_target(df, horizon=1):
    """Crea target binario mirando el retorno futuro."""
    df = df.copy()
    df['future_return'] = df['close'].shift(-horizon) / df['close'] - 1
    df['target'] = (df['future_return'] > 0).astype(int)
    df = df.dropna().reset_index(drop=True)  # <--- eliminar NaN después
    return df


In [67]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df = normalize_structure(df)

    if 'open_time' in df.columns and 'close_time' in df.columns:
        df = convert_datetime_columns(df)
        df['bar_duration'] = (df['close_time'] - df['open_time']).dt.total_seconds()
    else:
        df['bar_duration'] = np.nan

    if 'return' not in df.columns or df['return'].isna().all():
        df['return'] = df['close'].pct_change()

    # === Features derivadas ===
    df['volatility_10'] = df['return'].rolling(10).std()
    df['rsi_14'] = ta.momentum.RSIIndicator(df['close'], window=14).rsi()
    df['MA10_diff'] = (df['close'] - df['close'].rolling(10).mean()) / df['close'].rolling(10).mean()
    df['volume_z'] = (df['volume'] - df['volume'].rolling(20).mean()) / df['volume'].rolling(20).std()
    df['momentum_5'] = df['close'] / df['close'].shift(5) - 1
    df['h1_range'] = (df['high'] - df['low']) / df['close'].shift(1)
    df['oc_range'] = (df['close'] - df['open']) / df['open']
    df['volume_ma_ratio'] = df['volume'] / df['volume'].rolling(20).mean()

    for lag in [1, 2, 3]:
        df[f'return_lag_{lag}'] = df['return'].shift(lag)
        df[f'volume_lag_{lag}'] = df['volume'].shift(lag)

    # Target: retorno de la próxima barra (futuro)
    df['target'] = (df['return'].shift(-1) > 0).astype(int)

    # Eliminar columnas temporales
    df = df.drop(columns=[c for c in ['open_time', 'close_time'] if c in df.columns])

    df = df.dropna()

    return df


In [68]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report

In [69]:
def train_random_forest(df: pd.DataFrame, target_col: str = 'target', split_ratio=0.8, random_state=42):
    """
    Entrena un Random Forest en un dataset procesado con build_features().
    target_col: nombre de la columna objetivo (por ejemplo, 'target' o 'return_next')
    """
    # eliminar filas con NaN en el target
    df = df.dropna(subset=[target_col])

    # dividir train/test temporalmente
    split_point = int(len(df) * split_ratio)
    X_train = df.iloc[:split_point].drop(columns=[target_col])
    y_train = df.iloc[:split_point][target_col]
    X_test = df.iloc[split_point:].drop(columns=[target_col])
    y_test = df.iloc[split_point:][target_col]

    # entrenar modelo
    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        min_samples_split=10,
        min_samples_leaf=5,
        n_jobs=-1,
        random_state=random_state
    )

    model.fit(X_train, y_train)

    # predicciones
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if len(np.unique(y_test)) == 2 else None

    # métricas
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_prob) if y_prob is not None else np.nan

    print("📊 Resultados del modelo:")
    print(f"Accuracy: {acc:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"ROC-AUC:  {roc:.4f}")
    print("\nReporte completo:")
    print(classification_report(y_test, y_pred))

    return model, {"accuracy": acc, "f1": f1, "roc_auc": roc}


In [70]:
# === Construir features ===
time_bars_feat = build_features(original_data)
volume_bars_feat = build_features(volume_data)
dollar_bars_feat = build_features(dolar_data)

# === Entrenar ===
results = {}
for name, df in {
    'Time Bars': time_bars_feat,
    'Volume Bars': volume_bars_feat,
    'Dollar Bars': dollar_bars_feat
}.items():
    print(f"\n🚀 Entrenando modelo para {name}...")
    model, metrics = train_random_forest(df)
    results[name] = metrics


frac NaT tras parse directo: 0.000
frac NaT tras parse directo: 0.000
open_time -> NaT: 0/43138 (0.000%)
close_time -> NaT: 0/43138 (0.000%)
Conversión final: dtype open_time => datetime64[ns]


,open_time,close_time,open,high,low,close,volume,return
0,2024-12-01 05:01:00,2024-12-01 05:01:59.999,96473.19,96473.19,96464.16,96464.97,2.51941,-0.000085
1,2024-12-01 05:02:00,2024-12-01 05:02:59.999,96464.97,96509.99,96425.70,96509.99,40.57111,0.000467
2,2024-12-01 05:03:00,2024-12-01 05:03:59.999,96509.99,96510.00,96476.00,96480.00,6.33996,-0.000311
3,2024-12-01 05:04:00,2024-12-01 05:04:59.999,96480.01,96480.01,96472.00,96472.00,2.02027,-0.000083
4,2024-12-01 05:05:00,2024-12-01 05:05:59.999,96472.00,96472.01,96415.65,96420.02,8.16665,-0.000539


frac NaT tras parse directo: 0.000
frac NaT tras parse directo: 0.000
open_time -> NaT: 0/5286 (0.000%)
close_time -> NaT: 0/5286 (0.000%)
Conversión final: dtype open_time => datetime64[ns]


,open_time,close_time,open,high,low,close,volume,return
0,2024-12-01 05:01:00,2024-12-01 05:23:59.999,96473.19,96510.00,96385.47,96417.05,154.01150,NaN
1,2024-12-01 05:24:00,2024-12-01 05:53:59.999,96417.04,96468.66,96355.00,96400.00,142.63492,-0.000177
2,2024-12-01 05:54:00,2024-12-01 06:20:59.999,96400.00,96483.98,96284.51,96388.01,141.12160,-0.000124
3,2024-12-01 06:21:00,2024-12-01 06:50:59.999,96388.01,96464.00,96341.88,96355.00,145.17075,-0.000342
4,2024-12-01 06:51:00,2024-12-01 07:23:59.999,96355.00,96459.77,96299.10,96449.48,141.15671,0.000981


frac NaT tras parse directo: 0.000
frac NaT tras parse directo: 0.000
open_time -> NaT: 0/29397 (0.000%)
close_time -> NaT: 0/29397 (0.000%)
Conversión final: dtype open_time => datetime64[ns]


,open_time,close_time,open,high,low,close,volume,return
0,2024-12-01 05:01:00,2024-12-01 05:02:59.999,96473.19,96509.99,96425.70,96509.99,43.09052,0.000382
1,2024-12-01 05:03:00,2024-12-01 05:06:59.999,96509.99,96510.00,96415.65,96459.11,21.81001,-0.000527
2,2024-12-01 05:07:00,2024-12-01 05:12:59.999,96459.11,96470.50,96409.54,96409.57,22.95800,-0.000514
3,2024-12-01 05:13:00,2024-12-01 05:17:59.999,96409.56,96460.00,96385.47,96460.00,21.20811,0.000523
4,2024-12-01 05:18:00,2024-12-01 05:22:59.999,96460.00,96500.00,96428.57,96457.11,29.25800,-0.000030



🚀 Entrenando modelo para Time Bars...
📊 Resultados del modelo:
Accuracy: 0.5031
F1 Score: 0.5094
ROC-AUC:  0.5052

Reporte completo:
              precision    recall  f1-score   support

           0       0.51      0.48      0.50      4406
           1       0.49      0.53      0.51      4218

    accuracy                           0.50      8624
   macro avg       0.50      0.50      0.50      8624
weighted avg       0.50      0.50      0.50      8624


🚀 Entrenando modelo para Volume Bars...
📊 Resultados del modelo:
Accuracy: 0.5038
F1 Score: 0.5904
ROC-AUC:  0.5097

Reporte completo:
              precision    recall  f1-score   support

           0       0.49      0.30      0.37       518
           1       0.51      0.70      0.59       536

    accuracy                           0.50      1054
   macro avg       0.50      0.50      0.48      1054
weighted avg       0.50      0.50      0.48      1054


🚀 Entrenando modelo para Dollar Bars...
📊 Resultados del modelo:
Accuracy: 